# 01 — Data Extraction
**MediScope · DVA Capstone Project**

**Goal:** Load the five raw MIMIC-III CSV files, inspect their structure, validate
relational integrity, and persist copies to `data/processed/` so that downstream
notebooks (`02_cleaning`, `03_eda`, …) can consume them from a single stable location.

| Notebook | Role |
|---|---|
| **01_extraction** (this) | Load → Inspect → Save to processed/ |
| 02_cleaning | Clean & engineer features |
| 03_eda | Exploratory analysis & visualisations |
| 04_statistical_analysis | Hypothesis tests & models |
| 05_final_load_prep | Merge & export Tableau-ready CSVs |

## 1. Imports & Path Configuration

In [ ]:
import os
import pandas as pd
import numpy as np

# ── Resolve paths relative to this notebook's location ──────────────────────
NOTEBOOK_DIR  = os.path.dirname(os.path.abspath('__file__'))  # notebooks/
ROOT_DIR      = os.path.join(NOTEBOOK_DIR, '..')              # project root
RAW_DIR       = os.path.join(ROOT_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(ROOT_DIR, 'data', 'processed')

os.makedirs(PROCESSED_DIR, exist_ok=True)

print('RAW_DIR      :', os.path.abspath(RAW_DIR))
print('PROCESSED_DIR:', os.path.abspath(PROCESSED_DIR))

# ── Display settings ─────────────────────────────────────────────────────────
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows',    10)
pd.set_option('display.float_format', '{:.2f}'.format)

## 2. Dataset Manifest

All five source files come from a de-identified MIMIC-III subset.

| File | Description | Primary Key |
|---|---|---|
| `PATIENTS.csv` | Demographics (100 patients) | `subject_id` |
| `ADMISSIONS.csv` | Hospital stays (129 admissions) | `hadm_id` |
| `LABEVENTS.csv` | Lab test results (76,074 rows) | `(subject_id, itemid, charttime)` |
| `D_LABITEMS.csv` | Lab test code dictionary (753 items) | `itemid` |
| `structured_medical_records.csv` | Pre-structured records (408 rows) | — |

## 3. Load Raw Data

In [ ]:
FILE_MAP = {
    'patients'   : 'PATIENTS.csv',
    'admissions' : 'ADMISSIONS.csv',
    'labevents'  : 'LABEVENTS.csv',
    'labitems'   : 'D_LABITEMS.csv',
    'structured' : 'structured_medical_records.csv',
}

dfs = {}
for key, filename in FILE_MAP.items():
    path = os.path.join(RAW_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f'Missing file: {path}')
    dfs[key] = pd.read_csv(path, low_memory=False)
    print(f'  Loaded {filename:<42}  shape={dfs[key].shape}')

patients   = dfs['patients']
admissions = dfs['admissions']
labevents  = dfs['labevents']
labitems   = dfs['labitems']
structured = dfs['structured']

## 4. Per-Table Inspection

### 4.1 PATIENTS

In [ ]:
print('── PATIENTS ──')
print('Shape :', patients.shape)
patients.info()
patients.head(3)

In [ ]:
print('Missing values (PATIENTS):')
print(patients.isnull().sum())
print('\nUnique subject_id:', patients['subject_id'].nunique())

### 4.2 ADMISSIONS

In [ ]:
print('── ADMISSIONS ──')
print('Shape :', admissions.shape)
admissions.info()
admissions.head(3)

In [ ]:
print('Missing values (ADMISSIONS):')
print(admissions.isnull().sum())
print('\nUnique hadm_id        :', admissions['hadm_id'].nunique())
print('Unique admission_type :', admissions['admission_type'].unique())
print('Admissions per patient (describe):')
print(admissions.groupby('subject_id')['hadm_id'].count().describe())

### 4.3 LABEVENTS

In [ ]:
print('── LABEVENTS ──')
print('Shape :', labevents.shape)
labevents.info()
labevents.head(3)

In [ ]:
print('Missing values (LABEVENTS):')
print(labevents.isnull().sum())
print('\nUnique subjects   :', labevents['subject_id'].nunique())
print('Unique lab tests  :', labevents['itemid'].nunique())
print('Date range        :',
      labevents['charttime'].min(), '→', labevents['charttime'].max())

### 4.4 D_LABITEMS (Lab Test Dictionary)

In [ ]:
print('── D_LABITEMS ──')
print('Shape :', labitems.shape)
labitems.info()
labitems.head(3)

In [ ]:
print('Missing values (D_LABITEMS):')
print(labitems.isnull().sum())
# Top categories in the dictionary
print('\nTop fluid categories:')
print(labitems['fluid'].value_counts().head(10))

### 4.5 Structured Medical Records

In [ ]:
print('── STRUCTURED MEDICAL RECORDS ──')
print('Shape :', structured.shape)
structured.info()
structured.head(3)

In [ ]:
print('Missing values (STRUCTURED):')
print(structured.isnull().sum())
print('\nColumn names:', structured.columns.tolist())

## 5. Relational Integrity Checks

Verify foreign-key linkages before any merging happens in downstream notebooks.

In [ ]:
# ── 5a. Admissions → Patients (via subject_id) ───────────────────────────────
adm_subjects  = set(admissions['subject_id'])
pat_subjects  = set(patients['subject_id'])
missing_in_patients = adm_subjects - pat_subjects
print(f'Subjects in ADMISSIONS not in PATIENTS: {len(missing_in_patients)}')

# ── 5b. LabEvents → Patients (via subject_id) ────────────────────────────────
lab_subjects = set(labevents['subject_id'])
missing_lab  = lab_subjects - pat_subjects
print(f'Subjects in LABEVENTS not in PATIENTS : {len(missing_lab)}')

# ── 5c. LabEvents → D_LABITEMS (via itemid) ──────────────────────────────────
lab_items_set  = set(labitems['itemid'])
event_items    = set(labevents['itemid'])
unmatched_items = event_items - lab_items_set
print(f'LABEVENTS itemids not in D_LABITEMS   : {len(unmatched_items)}')

# ── 5d. Coverage: patients with ≥1 lab result ────────────────────────────────
patients_with_labs = lab_subjects & pat_subjects
print(f'Patients with ≥1 lab result           : {len(patients_with_labs)} / {len(pat_subjects)}')

## 6. Dataset Summary Dashboard

In [ ]:
summary = pd.DataFrame({
    'Table'      : ['PATIENTS', 'ADMISSIONS', 'LABEVENTS', 'D_LABITEMS', 'STRUCTURED'],
    'Rows'       : [patients.shape[0], admissions.shape[0], labevents.shape[0],
                    labitems.shape[0], structured.shape[0]],
    'Columns'    : [patients.shape[1], admissions.shape[1], labevents.shape[1],
                    labitems.shape[1], structured.shape[1]],
    'Null Cells' : [
        patients.isnull().sum().sum(),
        admissions.isnull().sum().sum(),
        labevents.isnull().sum().sum(),
        labitems.isnull().sum().sum(),
        structured.isnull().sum().sum(),
    ],
    'Primary Key': ['subject_id', 'hadm_id', '(subject_id, itemid, charttime)',
                    'itemid', '—'],
})
summary

## 7. Save Copies to `data/processed/`

Downstream notebooks (`02_cleaning.ipynb` onward) read from `data/processed/`
so that the raw files are never mutated.

In [ ]:
SAVE_MAP = {
    'patients_raw.csv'   : patients,
    'admissions_raw.csv' : admissions,
    'labevents_raw.csv'  : labevents,
    'labitems_raw.csv'   : labitems,
    'structured_raw.csv' : structured,
}

for fname, df in SAVE_MAP.items():
    out_path = os.path.join(PROCESSED_DIR, fname)
    df.to_csv(out_path, index=False)
    print(f'  Saved → {os.path.relpath(out_path)}')

print('\n✅ Extraction complete. data/processed/ is ready for 02_cleaning.ipynb.')

## 8. Extraction Sign-off

| Metric | Value |
|---|---|
| Total raw records loaded | **76,934 + 408** (MIMIC tables + structured) |
| Unique patients | **100** |
| Unique admissions | **129** |
| Lab event records | **76,074** |
| Lab test types (dictionary) | **753** |
| Files written to processed/ | **5** |

**Next step →** Open `02_cleaning.ipynb` to handle null imputation, datetime parsing, and feature engineering.